# Pilot Run: 100 銘柄 × ヒストリカル全期間

ランダムサンプリングした 100 社に対して、Strategy C (DOM table 除外 + ヒューリスティック subsection + paragraph packing) を **2002 年以降の全 10-K + 10-Q** に適用する。

## 設計

- **universe**: `/Volumes/personal_folder/Quants/FILING_NLP_v2/universe/universe_v2.parquet` (NYSE+Nasdaq 6,000 社)
- **サンプル**: seed=42 で 100 社抽出
- **filter**: form ∈ ('10-K', '10-Q')、filing_date >= 2002-01-01、amendment 除外
- **並列**: ThreadPoolExecutor(workers=8) + token bucket rate limiter (7 req/sec)
- **進捗**: tqdm.auto で表示
- **出力 (修正後)**: **per-CIK parquet**
    - `sections/pilot100/sections_cik0000056047.parquet` 形式
    - read-modify-write を回避し、kill 耐性と速度を両立
- **チェックポイント**: 既処理 CIK は skip して resume 可


In [14]:
# Cell 1: imports + edgar identity 設定
import logging
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / ".env").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        break
    REPO_ROOT = REPO_ROOT.parent
print(f"REPO_ROOT: {REPO_ROOT}")

env_path = REPO_ROOT / ".env"
if env_path.exists() and not os.environ.get("EDGAR_IDENTITY"):
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line.startswith("EDGAR_IDENTITY="):
            os.environ["EDGAR_IDENTITY"] = (
                line.split("=", 1)[1].strip().strip('"').strip("'")
            )
            break

import edgar

edgar.set_identity(os.environ["EDGAR_IDENTITY"])
print(f"EDGAR_IDENTITY: {os.environ['EDGAR_IDENTITY']}")


class _LegacyParserFilter(logging.Filter):
    def filter(self, record):
        return "falling back to legacy parser" not in record.getMessage()


logging.getLogger("edgar.core").addFilter(_LegacyParserFilter())

for name in ["httpx", "httpxthrottlecache", "httpcore", "edgar.documents"]:
    logging.getLogger(name).setLevel(logging.WARNING)

sys.path.insert(0, str(REPO_ROOT))

# モジュールキャッシュをクリア (runner.py の変更を反映)
for mod_name in list(sys.modules):
    if mod_name.startswith("notebook.FILING_NLP.pipeline"):
        del sys.modules[mod_name]

from notebook.FILING_NLP.pipeline import config, runner

print(f"NAS_ROOT: {config.NAS_ROOT}")
print(f"NAS exists: {config.NAS_ROOT.exists()}")

REPO_ROOT: /Users/yukihata/Desktop/quants
EDGAR_IDENTITY: YH-05 youxitiancore@gmail.com
NAS_ROOT: /Volumes/personal_folder/Quants/FILING_NLP_v2
NAS exists: True


In [15]:
# Cell 2: universe ロード + ランダムサンプリング
import pandas as pd

universe = pd.read_parquet(config.UNIVERSE_PARQUET)
print(f"universe: {len(universe):,} 社")
print(f"  exchange 分布:\n{universe['exchange'].value_counts()}")

SAMPLE_N = 100
SEED = 42
sample = universe.sample(n=SAMPLE_N, random_state=SEED).reset_index(drop=True)
print(f"\nsample: {len(sample)} 社 (seed={SEED})")
print(sample[["cik", "ticker", "exchange", "company"]].head(10).to_string(index=False))

universe: 6,000 社
  exchange 分布:
exchange
Nasdaq    3392
NYSE      2608
Name: count, dtype: int64

sample: 100 社 (seed=42)
    cik ticker exchange                          company
1089907   SWKH   Nasdaq                SWK Holdings Corp
1698530   XCUR   Nasdaq                    EXICURE, INC.
  56047    KEX     NYSE                       KIRBY CORP
1237831   GMED     NYSE               GLOBUS MEDICAL INC
1907982   QBTS     NYSE              D-Wave Quantum Inc.
 896878   INTU   Nasdaq                      INTUIT INC.
 830271    NMI     NYSE NUVEEN MUNICIPAL INCOME FUND INC
  36966    FHN     NYSE               FIRST HORIZON CORP
1057060    HZO     NYSE                    MARINEMAX INC
  78890    BCO     NYSE                        BRINKS CO


In [16]:
# Cell 3: gte-Qwen2 tokenizer ロード
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    config.TOKENIZER_MODEL_ID, trust_remote_code=True
)
print(f"tokenizer: {type(tokenizer).__name__}")
print(f"vocab_size: {tokenizer.vocab_size}")

tokenizer: Qwen2TokenizerFast
vocab_size: 151643


In [17]:
# Cell 4: 実行パラメータ + 既存 checkpoint 確認
RUN_ID = "pilot100"
WORKERS = 8
RATE_RPS = 7.0

sections_dir = config.NAS_ROOT / "sections" / RUN_ID
chunks_dir = config.NAS_ROOT / "chunks" / RUN_ID
filings_dir = config.NAS_ROOT / "filings_metadata" / RUN_ID
checkpoint_path = config.NAS_ROOT / "checkpoints" / f"{RUN_ID}_progress.json"
errors_path = config.NAS_ROOT / "logs" / f"{RUN_ID}_errors.jsonl"

print(f"run_id:       {RUN_ID}")
print(f"workers:      {WORKERS}")
print(f"rate_rps:     {RATE_RPS}")
print(f"sections_dir: {sections_dir}")
print(f"chunks_dir:   {chunks_dir}")
print(f"checkpoint:   {checkpoint_path}")

import json

if checkpoint_path.exists():
    cp = json.loads(checkpoint_path.read_text())
    print(f"\n既存 checkpoint: {len(cp.get('completed', {}))} CIK 完了 (resume)")
else:
    print("\n新規 run")

run_id:       pilot100
workers:      8
rate_rps:     7.0
sections_dir: /Volumes/personal_folder/Quants/FILING_NLP_v2/sections/pilot100
chunks_dir:   /Volumes/personal_folder/Quants/FILING_NLP_v2/chunks/pilot100
checkpoint:   /Volumes/personal_folder/Quants/FILING_NLP_v2/checkpoints/pilot100_progress.json

既存 checkpoint: 100 CIK 完了 (resume)


In [18]:
# Cell 5: RESET — 既存出力を全削除して fresh start
# ⚠️ 前回の bug (mark_done が flush 前) の影響で、既存 checkpoint には
# 「完了マーク済みだがデータなし」の CIK が含まれる可能性が高い。
# 修正後 (per-CIK 書き込み) で fresh run することを推奨。
RESET = True  # ← 必要に応じて False に変更

if RESET:
    import shutil

    for path in [sections_dir, chunks_dir, filings_dir]:
        if path.exists():
            shutil.rmtree(path)
            print(f"deleted: {path}")
    for path in [checkpoint_path, errors_path]:
        if path.exists():
            path.unlink()
            print(f"deleted: {path}")
    print("\n✅ fresh start ready")
else:
    print("RESET=False → 既存 checkpoint から resume")

deleted: /Volumes/personal_folder/Quants/FILING_NLP_v2/sections/pilot100
deleted: /Volumes/personal_folder/Quants/FILING_NLP_v2/chunks/pilot100
deleted: /Volumes/personal_folder/Quants/FILING_NLP_v2/filings_metadata/pilot100
deleted: /Volumes/personal_folder/Quants/FILING_NLP_v2/checkpoints/pilot100_progress.json
deleted: /Volumes/personal_folder/Quants/FILING_NLP_v2/logs/pilot100_errors.jsonl

✅ fresh start ready


In [19]:
# Cell 6: パイプライン実行 (tqdm 進捗 + per-CIK 書き込み)
summary = runner.run_pipeline(
    universe=sample,
    tokenizer=tokenizer,
    sections_dir=sections_dir,
    chunks_dir=chunks_dir,
    filings_metadata_dir=filings_dir,
    checkpoint_path=checkpoint_path,
    errors_path=errors_path,
    max_workers=WORKERS,
    rate_rps=RATE_RPS,
    use_tqdm=True,
)

print("\n=== 実行サマリ ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

CIKs:   0%|          | 0/100 [00:00<?, ?cik/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (33895 > 32768). Running this sequence through the model will result in indexing errors
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections found
All detection strategies failed, no sections foun


=== 実行サマリ ===
  n_processed: 100
  n_failed: 0
  n_filings: 4040
  n_sections: 7893
  n_chunks: 137970
  started_at: 2026-05-25T11:51:27.749094
  finished_at: 2026-05-25T13:27:57.630417
  elapsed_sec: 5789.9


In [20]:
# Cell 7: per-CIK parquet から集計


def _load_dir(dir_path: Path) -> pd.DataFrame:
    files = sorted(dir_path.glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)


all_chunks = _load_dir(chunks_dir)
all_sections = _load_dir(sections_dir)
all_filings = _load_dir(filings_dir)

print("=== 出力ファイル (per-CIK 方式) ===")
print(
    f"  sections files: {len(list(sections_dir.glob('*.parquet')))} (total {len(all_sections):,} rows)"
)
print(
    f"  chunks files:   {len(list(chunks_dir.glob('*.parquet')))} (total {len(all_chunks):,} rows)"
)
print(
    f"  filings files:  {len(list(filings_dir.glob('*.parquet')))} (total {len(all_filings):,} rows)"
)

print("\n=== ユニーク銘柄 ===")
print(
    f"  unique CIK in chunks: {all_chunks['cik'].nunique() if len(all_chunks) else 0}"
)
print(
    f"  unique ticker:        {all_chunks['ticker'].nunique() if len(all_chunks) else 0}"
)

print("\n=== fiscal_year 範囲 ===")
if len(all_chunks):
    print(f"  {all_chunks['fiscal_year'].min()} - {all_chunks['fiscal_year'].max()}")

print("\n=== form × section_key 分布 ===")
if len(all_chunks):
    print(
        all_chunks.groupby(["form", "section_key", "section_role"])
        .size()
        .to_frame("chunks")
    )

print("\n=== token_count 統計 ===")
if len(all_chunks):
    print(all_chunks["token_count"].describe().round(1).to_string())
    over = (all_chunks["token_count"] > config.MAX_TOKENS).sum()
    print(
        f"  上限超過 (>{config.MAX_TOKENS}): {over} ({100 * over / len(all_chunks):.2f}%)"
    )

=== 出力ファイル (per-CIK 方式) ===
  sections files: 66 (total 7,893 rows)
  chunks files:   66 (total 137,970 rows)
  filings files:  66 (total 4,040 rows)

=== ユニーク銘柄 ===
  unique CIK in chunks: 66
  unique ticker:        66

=== fiscal_year 範囲 ===
  2002 - 2026

=== form × section_key 分布 ===
                               chunks
form section_key section_role        
10-K item_1      business       21474
     item_1a     risk_factors   17216
     item_7      mda            27989
10-Q item_1a     risk_factors    7782
     item_2      mda            63509

=== token_count 統計 ===
count    137970.0
mean        380.1
std         360.6
min           1.0
25%          73.0
50%         230.0
75%         715.0
max        3541.0
  上限超過 (>1024): 721 (0.52%)


In [21]:
# Cell 8: データ品質分析
print("=== filings 取得状況 by status ===")
if len(all_filings):
    print(all_filings["status"].value_counts())

print("\n=== DOM section 取得成功率 by section_key ===")
if len(all_sections):
    dom_stats = all_sections.groupby("section_key").agg(
        n=("filing_id", "count"),
        dom_found=("dom_section_found", "sum"),
        tables_removed_mean=("tables_removed", "mean"),
        text_len_mean=("char_count", "mean"),
    )
    dom_stats["dom_rate"] = (dom_stats["dom_found"] / dom_stats["n"] * 100).round(1)
    print(dom_stats.round(1))

print("\n=== chunks がゼロだった CIK ===")
if len(all_filings):
    cik_chunks = (
        all_chunks.groupby(["cik", "ticker"]).size().to_frame("n_chunks")
        if len(all_chunks)
        else pd.DataFrame()
    )
    sample_with_chunks = sample.merge(
        cik_chunks, left_on="cik", right_on="cik", how="left"
    ).fillna(0)
    zero_chunk = sample_with_chunks[sample_with_chunks["n_chunks"] == 0]
    print(f"  {len(zero_chunk)} 社 が chunks ゼロ")
    if len(zero_chunk):
        print(
            zero_chunk[["cik", "ticker", "exchange", "company"]]
            .head(10)
            .to_string(index=False)
        )

print("\n=== errors.jsonl 集計 ===")
if errors_path.exists():
    import json as _json

    errs = []
    with errors_path.open() as fp:
        for line in fp:
            try:
                errs.append(_json.loads(line))
            except Exception:
                continue
    print(f"  total errors logged: {len(errs)}")
    if errs:
        phase_dist = pd.Series([e.get("phase", "unknown") for e in errs]).value_counts()
        print(f"  phase 分布:\n{phase_dist}")
else:
    print("  (no errors logged)")

=== filings 取得状況 by status ===
status
success        3781
no_sections     259
Name: count, dtype: int64

=== DOM section 取得成功率 by section_key ===
                n  dom_found  tables_removed_mean  text_len_mean  dom_rate
section_key                                                               
item_1        949        353                  1.6        47905.9      37.2
item_1a      3230       1855                  2.0        26690.8      57.4
item_2       2778       2161                  7.1        50416.0      77.8
item_7        936        348                  2.1        70458.1      37.2

=== chunks がゼロだった CIK ===
  34 社 が chunks ゼロ
    cik ticker exchange                                          company
 830271    NMI     NYSE                 NUVEEN MUNICIPAL INCOME FUND INC
1938865    WAI   Nasdaq                                  Top KingWin Ltd
1395213    EDN     NYSE                                           EDENOR
 810766    CIK     NYSE CREDIT SUISSE ASSET MANAGEMENT INCOME FUND

In [22]:
# Cell 9: ticker × year の chunks 分布
if len(all_chunks):
    pivot = all_chunks.pivot_table(
        index="ticker",
        columns="fiscal_year",
        values="chunk_idx",
        aggfunc="count",
        fill_value=0,
    )
    print(f"ticker × fiscal_year (chunk 数): shape={pivot.shape}")
    print(pivot.head(15).to_string())

if len(all_chunks):
    print("\n=== fiscal_year 別 chunks 数 ===")
    print(all_chunks.groupby("fiscal_year").size().to_string())

ticker × fiscal_year (chunk 数): shape=(66, 25)
fiscal_year  2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
ticker                                                                                                                                                           
AA              0     0     0     0     0     0     0     0     0     0     0     0     0     0    24   179   130   110   439   402   362   376   279   264   167
ACRS            0     0     0     0     0     0     0     0     0     0     0     0     0    58   144   147   138   111   101   124   126   109   107   122    92
ADUS            0     0     0     0     0     0     0    15    72    94   103   106    98   114   103   110   134    52   462   450   443   319   320   340   253
AENT            0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0    91   179   305